In [ ]:
# -*- coding: utf-8 -*-
"""
baseline_manager.py - offline PubMed master database builder (pipeline Step 0).

Downloads the complete NLM PubMed Baseline (and optional daily update files)
over FTP and compiles them into a single local SQLite "master" database of
PMIDs, publication dates, and MeSH annotations that the rest of the pipeline
queries entirely offline.

Parsing is parallelized across CPU cores with a MapReduce sharding pattern: each
worker parses a chunk of gzipped XML into its own temporary SQLite shard, and
the shards are merged into the master database. To stay safe on local disks,
network-attached storage, and cloud-synced folders, the build runs in a fast
local workspace and copies back with SHA-256 checksum verification and periodic
atomic checkpoints, so an interrupted run can resume.
"""

import os
import ftplib
import gzip
import sqlite3
import time
import shutil
import tempfile
import hashlib
import multiprocessing
import xml.etree.ElementTree as ET
from pathlib import Path
from uuid import uuid4

# ==========================================
# MODULE-LEVEL FUNCTIONS (Required for OS-Agnostic Multiprocessing)
# ==========================================

def _force_os_sync():
    """Forces the OS to flush write buffers to the physical disk (OS-Agnostic)."""
    if hasattr(os, 'sync'):
        os.sync()

def get_file_hash(filepath):
    """Calculates the SHA-256 digital fingerprint of a file in low-memory blocks."""
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            hasher.update(chunk)
    return hasher.hexdigest()

def verified_safe_transfer(src_path: Path, dest_path: Path, max_retries=5):
    """Copies a file and strictly verifies the copy is identical byte-for-byte."""
    print(f"      [Sync] Profiling local database fingerprint...", end=" ", flush=True)
    src_size = os.path.getsize(src_path)
    src_hash = get_file_hash(src_path)
    print("Done.")

    tmp_dest = Path(str(dest_path) + ".tmp")

    for attempt in range(1, max_retries + 1):
        print(f"      [Sync] Attempt {attempt}/{max_retries} - Transferring to target storage...", end=" ", flush=True)
        try:
            if tmp_dest.exists():
                tmp_dest.unlink()

            shutil.copy2(src_path, tmp_dest)
            _force_os_sync()

            dest_size = os.path.getsize(tmp_dest)
            if dest_size != src_size:
                print(f"FAIL (Size mismatch: {dest_size} != {src_size})")
                continue

            dest_hash = get_file_hash(tmp_dest)
            if src_hash == dest_hash:
                os.replace(tmp_dest, dest_path)
                print("SUCCESS (Cryptographic Match Verified).")
                return True
            else:
                print(f"FAIL (Hash mismatch detected).")
                continue

        except Exception as e:
            print(f"ERROR ({e})")
            time.sleep(5)

    raise Exception("\n[CRITICAL ERROR] Failed to transfer a healthy database after maximum retries.")

def _extract_pub_date(elem) -> str:
    """Safely extracts and formats the publication date into YYYY-MM-DD."""
    # 1. Try to get the official PubMed indexing date
    pub_date_node = elem.find('.//PubMedPubDate[@PubStatus="pubmed"]')
    if pub_date_node is not None:
        y = pub_date_node.findtext('Year')
        m = pub_date_node.findtext('Month')
        d = pub_date_node.findtext('Day')
        if y:
            m = m.zfill(2) if m else '01'
            d = d.zfill(2) if d else '01'
            return f"{y}-{m}-{d}"

    # 2. Fallback to Journal Issue PubDate
    pub_date_node = elem.find('.//PubDate')
    if pub_date_node is not None:
        y = pub_date_node.findtext('Year')
        m_str = pub_date_node.findtext('Month')
        d = pub_date_node.findtext('Day')

        if y:
            m = '01'
            if m_str:
                if m_str.isdigit():
                    m = m_str.zfill(2)
                else:
                    m_map = {
                        'jan':'01', 'feb':'02', 'mar':'03', 'apr':'04',
                        'may':'05', 'jun':'06', 'jul':'07', 'aug':'08',
                        'sep':'09', 'oct':'10', 'nov':'11', 'dec':'12'
                    }
                    m = m_map.get(m_str.lower()[:3], '01')
            d = d.zfill(2) if d else '01'
            return f"{y}-{m}-{d}"

    # 3. Default fallback if absolutely no date is found
    return "1900-01-01"

def build_local_shard(local_file_chunk):
    """Executes on parallel CPU cores to parse XML chunks into local SQLite shards."""
    shard_path = Path(tempfile.gettempdir()) / f"shard_{uuid4().hex}.db"
    conn = sqlite3.connect(shard_path)

    conn.execute("PRAGMA journal_mode = OFF;")
    conn.execute("PRAGMA synchronous = OFF;")
    # UPDATED SCHEMA: Now includes pub_date
    conn.execute("CREATE TABLE shard_data (pmid INTEGER, pub_date TEXT, mesh_terms TEXT, source_file TEXT)")

    processed_names = []
    total_articles = 0
    cursor = conn.cursor()

    for filepath in local_file_chunk:
        batch = []
        source_filename = Path(filepath).name
        try:
            with gzip.open(filepath, 'rb') as f:
                # Track the root to drop parsed siblings; otherwise each ~30k-article
                # baseline file accumulates in memory until the whole file is done.
                context = ET.iterparse(f, events=('start', 'end'))
                _, root = next(context)
                for event, elem in context:
                    if event != 'end' or elem.tag != 'PubmedArticle':
                        continue

                    pmid_node = elem.find('.//PMID')
                    mesh_list = elem.find('.//MeshHeadingList')

                    if pmid_node is not None and mesh_list is not None:
                        pmid = int(pmid_node.text)
                        pub_date = _extract_pub_date(elem)
                        terms = [f"{'*' if d.get('MajorTopicYN') == 'Y' else ''}{d.text}"
                                 for d in mesh_list.findall('.//DescriptorName')]

                        if terms:
                            batch.append((pmid, pub_date, ";".join(terms), source_filename))

                    elem.clear()
                    root.clear()

            if batch:
                cursor.executemany("INSERT INTO shard_data (pmid, pub_date, mesh_terms, source_file) VALUES (?, ?, ?, ?)", batch)
                total_articles += len(batch)

            processed_names.append(source_filename)
        except Exception as e:
            print(f"\n  [!] Error parsing {source_filename}: {e}")

    conn.commit()
    conn.close()
    return str(shard_path), processed_names, total_articles


# ==========================================
# MAIN MANAGER CLASS
# ==========================================

class PubMedBaselineManager:
    """Downloads the NLM PubMed XML archives and compiles them into the local master database."""

    def __init__(self, raw_data_dir: Path, master_db_path: Path):
        """Set up the raw/download directories, fast local workspace, and FTP endpoints."""
        self.raw_data_dir = Path(raw_data_dir)
        self.master_db_path = Path(master_db_path)

        self.baseline_dir = self.raw_data_dir / "pubmed_baseline"
        self.updates_dir = self.raw_data_dir / "pubmed_updates"

        self.baseline_dir.mkdir(parents=True, exist_ok=True)
        self.updates_dir.mkdir(parents=True, exist_ok=True)

        # High-Speed Local Workspace (OS-Agnostic)
        self.local_workspace = Path(tempfile.gettempdir()) / "mesh_etl_workspace"
        self.local_workspace.mkdir(exist_ok=True)
        self.local_db_path = self.local_workspace / "local_active_master.db"
        self.local_xml_dir = self.local_workspace / "xml_buffer"
        self.local_xml_dir.mkdir(exist_ok=True)

        self.ftp_host = "ftp.ncbi.nlm.nih.gov"
        self.baseline_ftp_path = "/pubmed/baseline/"
        self.updates_ftp_path = "/pubmed/updatefiles/"

        self.chunk_size = 25
        self.checkpoint_interval = 4  # Execute backup every 100 files

    def _get_ftp_file_list(self, ftp_path: str) -> list:
        """Return the list of .xml.gz filenames available at the given NCBI FTP path."""
        try:
            ftp = ftplib.FTP(self.ftp_host)
            ftp.login()
            ftp.cwd(ftp_path)
            files = [f for f in ftp.nlst() if f.endswith(".xml.gz")]
            ftp.quit()
            return files
        except Exception as e:
            print(f"  [!] FTP Connection Error while listing files: {e}")
            return []

    def _download_files(self, ftp_path: str, local_dir: Path, file_list: list, limit: int = None):
        """Download the listed files to local_dir, skipping existing ones and reconnecting/retrying on failure."""
        files_to_download = file_list[:limit] if limit else file_list
        ftp = None

        def connect():
            nonlocal ftp
            if ftp:
                try:
                    ftp.quit()
                except:
                    pass
            ftp = ftplib.FTP(self.ftp_host)
            ftp.login()
            ftp.cwd(ftp_path)

        connect()
        print(f"  Starting download sequence for {len(files_to_download)} files...")

        for i, filename in enumerate(files_to_download, 1):
            local_filepath = local_dir / filename
            if local_filepath.exists() and local_filepath.stat().st_size > 1024:
                continue

            success = False
            for attempt in range(3):
                try:
                    with open(local_filepath, 'wb') as f:
                        ftp.retrbinary(f"RETR {filename}", f.write)
                    success = True
                    if i % 50 == 0 or i == len(files_to_download):
                        print(f"    -> Downloaded {i}/{len(files_to_download)} files...")
                    break
                except Exception as e:
                    print(f"    [!] Error on {filename} (Attempt {attempt+1}/3): {e}")
                    if local_filepath.exists():
                        local_filepath.unlink()
                    time.sleep(2)
                    connect()
            if not success:
                print(f"    [!] FAILED to download {filename}. Skipping.")

        if ftp:
            try:
                ftp.quit()
            except:
                pass

    def run_downloads(self, include_baseline=True, include_updates=False, limit=None):
        """Fetch the PubMed baseline and/or daily-update XML archives from NCBI over FTP."""
        print("\n" + "<"*30 + ">"*30)
        print("<<< NLM PubMed XML Data Download >>>")
        print("<"*30 + ">"*30)
        if include_baseline:
            print(f"  Fetching Baseline index from {self.ftp_host}...")
            baseline_files = self._get_ftp_file_list(self.baseline_ftp_path)
            if baseline_files:
                self._download_files(self.baseline_ftp_path, self.baseline_dir, baseline_files, limit)

        if include_updates:
            print(f"\n  Fetching Updates index from {self.ftp_host}...")
            update_files = self._get_ftp_file_list(self.updates_ftp_path)
            if update_files:
                self._download_files(self.updates_ftp_path, self.updates_dir, update_files, limit)

    def compile_database(self):
        """Parse all downloaded XML into the master SQLite database in parallel, with resumable checkpoints."""
        print("\n" + "<"*30 + ">"*30)
        print("<<< Phase 0: Environment Diagnostics & Bootstrapping >>>")
        print("<"*30 + ">"*30)

        os.makedirs(os.path.dirname(self.master_db_path), exist_ok=True)

        if self.master_db_path.exists():
            print("  Existing target database detected. Syncing to Local Workspace...")
            verified_safe_transfer(self.master_db_path, self.local_db_path)
        else:
            print("  No target database found. Initializing blank Local Workspace.")
            if self.local_db_path.exists():
                self.local_db_path.unlink()

        conn = sqlite3.connect(self.local_db_path)
        conn.execute("PRAGMA journal_mode = WAL;")

        # --- SCHEMA SELF-HEALING CHECK ---
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(master_mesh_annotations)")
        cols = cursor.fetchall()
        if cols and len(cols) < 4:
            print("  [!] Outdated 3-column schema detected. Purging and upgrading to 4-column schema...")
            conn.execute("DROP TABLE IF EXISTS master_mesh_annotations")
            conn.execute("DROP TABLE IF EXISTS parsed_files")

        # 4-column schema. pmid is the PRIMARY KEY so the INSERT OR REPLACE during
        # sharded load de-duplicates NLM baseline overlaps automatically and pmid
        # joins are indexed without a separate index build.
        conn.execute("CREATE TABLE IF NOT EXISTS master_mesh_annotations (pmid INTEGER PRIMARY KEY, pub_date TEXT, mesh_terms TEXT, source_file TEXT)")
        conn.execute("CREATE TABLE IF NOT EXISTS parsed_files (filename TEXT PRIMARY KEY)")

        cursor.execute("DROP INDEX IF EXISTS idx_pmid")
        cursor.execute("SELECT filename FROM parsed_files")
        completed_files = {row[0] for row in cursor.fetchall()}

        all_files = []
        if self.baseline_dir.exists():
            all_files.extend(list(self.baseline_dir.glob("*.xml.gz")))
        if self.updates_dir.exists():
            all_files.extend(list(self.updates_dir.glob("*.xml.gz")))

        pending_files = [f for f in all_files if f.name not in completed_files]

        if not pending_files:
            print("  All files parsed! Proceeding to Indexing Phase...")
        else:
            print(f"  Found {len(all_files)} total files. {len(completed_files)} already complete.")
            print(f"  Resuming with {len(pending_files)} files...")

            chunks = [pending_files[i:i + self.chunk_size] for i in range(0, len(pending_files), self.chunk_size)]
            cores = max(1, multiprocessing.cpu_count() - 1)

            print("\n" + "<"*30 + ">"*30)
            print(f"<<< Phase 1 & 2: Distributed Parsing & Aggregation ({cores} Cores) >>>")
            print("<"*30 + ">"*30)

            start_time = time.time()
            global_articles = 0

            # One worker pool for the whole run; re-creating it per chunk paid
            # process-spawn overhead on every block (dozens of times per ETL).
            pool = multiprocessing.Pool(cores)
            try:
                for chunk_idx, chunk in enumerate(chunks, 1):
                    local_chunk_paths = []
                    for raw_file in chunk:
                        local_dest = self.local_xml_dir / raw_file.name
                        shutil.copy2(raw_file, local_dest)
                        local_chunk_paths.append(local_dest)

                    sub_chunks = [local_chunk_paths[i::cores] for i in range(cores)]
                    sub_chunks = [sc for sc in sub_chunks if sc]

                    for shard_path_str, parsed_names, article_count in pool.imap_unordered(build_local_shard, sub_chunks):
                        cursor.execute("ATTACH DATABASE ? AS temp_shard", (shard_path_str,))
                        cursor.execute("BEGIN TRANSACTION;")

                        cursor.execute("INSERT OR REPLACE INTO master_mesh_annotations (pmid, pub_date, mesh_terms, source_file) SELECT pmid, pub_date, mesh_terms, source_file FROM temp_shard.shard_data")

                        for fname in parsed_names:
                            cursor.execute("INSERT OR REPLACE INTO parsed_files (filename) VALUES (?)", (fname,))

                        conn.commit()
                        cursor.execute("DETACH DATABASE temp_shard")
                        os.remove(shard_path_str)
                        global_articles += article_count

                    for local_file in local_chunk_paths:
                        if local_file.exists():
                            local_file.unlink()

                    cursor.execute("SELECT count(*) FROM parsed_files")
                    total_done = cursor.fetchone()[0]
                    elapsed = time.time() - start_time
                    print(f"  -> Processed Block {chunk_idx}/{len(chunks)}. (Total: {total_done}/{len(all_files)}) [+ {global_articles:,} articles] [{elapsed/60:.1f} min]")

                    # <<< ATOMIC CHECKPOINT >>>
                    if chunk_idx % self.checkpoint_interval == 0:
                        print(f"  <<< Executing Routine Checkpoint ({total_done} files) >>>")
                        conn.commit()
                        conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
                        conn.execute("PRAGMA journal_mode = DELETE;")
                        conn.close()

                        verified_safe_transfer(self.local_db_path, self.master_db_path)

                        conn = sqlite3.connect(self.local_db_path)
                        conn.execute("PRAGMA journal_mode = WAL;")
                        cursor = conn.cursor()
            finally:
                pool.close()
                pool.join()

        print("\n" + "<"*30 + ">"*30)
        print("<<< Phase 3: Post-Load Optimization & Indexing >>>")
        print("<"*30 + ">"*30)
        cursor.execute("SELECT count(*) FROM sqlite_master WHERE type='index' AND name='idx_pmid'")

        if cursor.fetchone()[0] == 0:
            print("  Cleaning duplicate records from NLM Baseline overlaps (This may take a few minutes)...")
            cursor.execute("""
                DELETE FROM master_mesh_annotations
                WHERE rowid NOT IN (
                    SELECT MIN(rowid)
                    FROM master_mesh_annotations
                    GROUP BY pmid
                )
            """)
            conn.commit()

            print("  Building B-Tree Index locally...")
            cursor.execute("DROP INDEX IF EXISTS idx_pmid")
            index_start = time.time()
            cursor.execute("CREATE UNIQUE INDEX idx_pmid ON master_mesh_annotations (pmid)")
            print(f"  Index complete. [{((time.time() - index_start)/60):.1f} min]")
        else:
            print("  Index already exists.")

        print("\n" + "<"*30 + ">"*30)
        print("<<< Phase 4: Final Network Transfer >>>")
        print("<"*30 + ">"*30)
        conn.commit()
        conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
        conn.execute("PRAGMA journal_mode = DELETE;")

        print("  Running final database defragmentation (VACUUM)...", end=" ", flush=True)
        conn.execute("VACUUM;")
        print("Done.")
        conn.close()

        verified_safe_transfer(self.local_db_path, self.master_db_path)

        print("\n" + "<"*30 + ">"*30)
        print("MASTER DATABASE COMPILATION COMPLETE")
        print("<"*30 + ">"*30 + "\n")